# Preparación de datos — Predicción de Popularidad de Canciones en Spotify
### MLY1101 Machine Learning — Caso C: Spotify Tracks

Este notebook sigue la metodología **CRISP-DM** y cubre:
1. Comprensión del negocio
2. Comprensión y preparación de los datos (incluye corrección de canciones repetidas por género)


## 1. Comprensión del negocio

**Problema de negocio:** una distribuidora/plataforma musical quiere anticipar qué tan popular será una canción a partir de sus características de audio y de su género, para priorizar qué temas promocionar y optimizar el presupuesto de marketing.

**Objetivo:** construir modelos de Machine Learning que predigan `popularity` (0-100).

**KPIs:**
- Error de predicción (RMSE / MAE sobre `popularity`).
- % de canciones "top" (popularity > 70) correctamente identificadas.

**Fuente de datos:** [Spotify Tracks Dataset (Kaggle)](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset). Herramienta colaborativa: GitHub.


## 2. Comprensión y preparación de los datos

### 2.1 Carga del dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/dataset.csv')

# 'Unnamed: 0' es solo un índice (0, 1, 2...) y no aporta información.
# Se elimina antes de buscar duplicados, porque si no cada fila parece distinta.
df = df.drop(columns=['Unnamed: 0'])

print('Dimensiones:', df.shape)
df.head()

Dimensiones: (114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


### 2.2 Evaluación de calidad de datos

In [2]:
print('Filas con algún nulo:', df.isna().any(axis=1).sum())
print('Filas 100% duplicadas:', df.duplicated().sum())
print('Canciones con duration_ms = 0:', (df['duration_ms'] == 0).sum())
print('Canciones con popularity = 0:', (df['popularity'] == 0).sum())

Filas con algún nulo: 1
Filas 100% duplicadas: 450
Canciones con duration_ms = 0: 1
Canciones con popularity = 0: 16020


In [3]:
# track_id repetidos: ¿son errores o la misma canción en varios géneros?
print('Filas:', len(df))
print('Canciones únicas (track_id):', df['track_id'].nunique())

generos_por_id = df.groupby('track_id')['track_genre'].nunique()
print('Canciones en más de un género:', (generos_por_id > 1).sum())
print('Máximo de géneros para una misma canción:', generos_por_id.max())

Filas: 114000
Canciones únicas (track_id): 89741
Canciones en más de un género: 16299
Máximo de géneros para una misma canción: 9


In [4]:
# Ejemplo: una misma canción repetida con distinto género
ejemplo = generos_por_id[generos_por_id > 1].index[0]
df[df['track_id'] == ejemplo][['track_id', 'track_name', 'track_genre', 'popularity', 'danceability']]

,track_id,track_name,track_genre,popularity,danceability
15028,001APMDOl3qtx1526T11n1,Better,chill,0,0.613
103211,001APMDOl3qtx1526T11n1,Better,soul,0,0.613


**Hallazgos de calidad:**
- 1 fila con nulos (artista, álbum y nombre) que además tiene `duration_ms = 0` (valor imposible). Es la misma fila.
- 450 filas 100% duplicadas.
- **Hay muchas más filas que canciones:** la misma canción aparece una vez por cada género en que está catalogada. Si se dejan así, una canción pesa 2, 3 o hasta 9 veces en el análisis y en el modelo (volumen inflado), y una misma canción podría caer en entrenamiento y en prueba a la vez (fuga de datos). Se corrige en 2.4.


### 2.3 Limpieza básica

In [5]:
df_clean = df.drop_duplicates()
print('Eliminadas por duplicado exacto:', len(df) - len(df_clean))

antes = len(df_clean)
df_clean = df_clean.dropna()
print('Eliminadas por nulos:', antes - len(df_clean))

antes = len(df_clean)
df_clean = df_clean[df_clean['duration_ms'] > 0]
print('Eliminadas por duration_ms = 0:', antes - len(df_clean))

print('Filas tras limpieza básica:', len(df_clean))

Eliminadas por duplicado exacto: 450
Eliminadas por nulos: 1
Eliminadas por duration_ms = 0: 0
Filas tras limpieza básica: 113549


### 2.4 Corrección: una fila por canción (multi-hot encoding de géneros)

Se deja **una sola fila por `track_id`**. Para no perder la información de género, cada género pasa a ser una columna con 0 o 1 (`genre_pop`, `genre_rock`, ...) y una canción en varios géneros tiene varios 1. Esto se llama *multi-hot encoding*.

Antes de juntar las filas hay que comprobar que el resto de las columnas sean iguales entre las filas repetidas de una misma canción.

In [6]:
otras = [c for c in df_clean.columns if c != 'track_genre']
varian = (df_clean.groupby('track_id')[otras].nunique() > 1).sum()
print('Canciones cuyas columnas cambian entre filas repetidas:')
print(varian[varian > 0])

Canciones cuyas columnas cambian entre filas repetidas:
popularity    720
dtype: int64


In [7]:
# Solo cambia 'popularity' en algunas canciones: ¿por cuánto?
rango = df_clean.groupby('track_id')['popularity'].agg(lambda s: s.max() - s.min())
rango = rango[rango > 0]
print('Canciones con popularity distinta entre sus filas:', len(rango))
print('Diferencia mediana:', rango.median(), '| diferencia máxima:', rango.max())

Canciones con popularity distinta entre sus filas: 720
Diferencia mediana: 1.0 | diferencia máxima: 44


Las audio features, el artista y el álbum son idénticos en todas las filas de una canción. Solo `popularity` cambia en 720 canciones, casi siempre por 1 punto. Se usa el **promedio** de `popularity` de esas filas.

In [8]:
# 1) Géneros: una columna 0/1 por género, combinadas por canción
generos = pd.get_dummies(df_clean['track_genre'], prefix='genre', dtype=int)
generos['track_id'] = df_clean['track_id'].values
generos = generos.groupby('track_id').max()

# 2) Resto de columnas: una fila por canción, con popularity promedio
popularidad = df_clean.groupby('track_id')['popularity'].mean().round().astype(int)
canciones = df_clean.drop(columns='track_genre').drop_duplicates(subset='track_id').set_index('track_id')
canciones['popularity'] = popularidad

# 3) Unir
df_final = canciones.join(generos).reset_index()
genre_cols = [c for c in df_final.columns if c.startswith('genre_')]

print('Dimensiones finales:', df_final.shape)
print('Canciones únicas:', df_final['track_id'].nunique())
print('Columnas de género:', len(genre_cols))

Dimensiones finales: (89740, 133)
Canciones únicas: 89740
Columnas de género: 114


In [9]:
# Comprobaciones
assert df_final['track_id'].is_unique                  # una fila por canción
assert (df_final[genre_cols].sum(axis=1) >= 1).all()   # toda canción tiene al menos un género

# No se perdió información de géneros: pares canción-género antes = unos después
pares_antes = df_clean[['track_id', 'track_genre']].drop_duplicates().shape[0]
print('Pares canción-género antes:', pares_antes, '| unos en las columnas de género:', df_final[genre_cols].sum().sum())

# Cuántos géneros tiene cada canción
df_final[genre_cols].sum(axis=1).value_counts().sort_index()

Pares canción-género antes: 113549 | unos en las columnas de género: 113549


1    73441
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: count, dtype: int64

### 2.4.1 Revisión de valores inválidos o extremos

Antes del EDA se revisan valores que no son posibles o que son extremos en las audio features: un `tempo` de 0 BPM o un `time_signature` de 0 no existen en una canción real, un `loudness` mayor a 0 dB es atípico en música comercial, y hay duraciones muy cortas o muy largas.

In [10]:
# Valores inválidos o extremos: se revisan, se documentan y se conservan (decisión en la celda siguiente)
chequeos = {
    'tempo = 0 BPM': df_final['tempo'] == 0,
    'time_signature = 0': df_final['time_signature'] == 0,
    'loudness > 0 dB': df_final['loudness'] > 0,
    'duración < 30 segundos': df_final['duration_ms'] < 30_000,
    'duración > 20 minutos': df_final['duration_ms'] > 20 * 60 * 1000,
}
resumen = pd.DataFrame({'Canciones': {k: int(v.sum()) for k, v in chequeos.items()}})
resumen['% del total'] = (resumen['Canciones'] / len(df_final) * 100).round(2)
print(resumen.to_string())

alguno = pd.concat(chequeos.values(), axis=1).any(axis=1)
print(f"\nCanciones con al menos uno de estos casos: {alguno.sum():,} ({alguno.mean():.2%})")
print('tempo = 0 y time_signature = 0 a la vez:', int(((df_final['tempo'] == 0) & (df_final['time_signature'] == 0)).sum()))
print(f"Duración máxima: {df_final['duration_ms'].max() / 60000:.0f} min | asimetría de duration_ms: {df_final['duration_ms'].skew():.1f}")

                        Canciones  % del total
tempo = 0 BPM                 157         0.17
time_signature = 0            162         0.18
loudness > 0 dB                68         0.08
duración < 30 segundos         14         0.02
duración > 20 minutos          73         0.08

Canciones con al menos uno de estos casos: 313 (0.35%)
tempo = 0 y time_signature = 0 a la vez: 157
Duración máxima: 87 min | asimetría de duration_ms: 11.1


**Decisión: se conservan y se documentan, no se eliminan.**
- Son pocos: en conjunto afectan a ≈ 0,35% de las canciones (313 canciones) y cada caso individual está bajo 0,5%.
- Las 157 canciones con `tempo = 0` son las mismas que tienen `time_signature = 0`: parecen fallos del análisis automático de audio de Spotify (no son valores posibles); no se pueden corregir con los datos disponibles.
- Gradient Boosting divide por umbrales, así que es poco sensible a valores extremos; en la regresión lineal el efecto de unos pocos puntos entre ~90.000 es pequeño; y en el clustering la robustez a valores extremos se evalúa en `04_clustering.ipynb` (transformaciones log y Yeo-Johnson).
- Eliminar filas cambiaría los datos limpios que usan los notebooks 02, 03 y 04 por un efecto marginal.

**Limitación:** el resultado puede tener algo de ruido por estos casos; no se espera que cambie las conclusiones.

### 2.5 Resumen y guardado

In [11]:
print(f'Filas originales:            {len(df):>7}')
print(f'Tras limpieza básica:         {len(df_clean):>7}')
print(f'Canciones únicas (final):     {len(df_final):>7}')

df_final.to_csv('../data/processed/spotify_tracks_clean.csv', index=False)
df_final[['track_id', 'track_name', 'popularity'] + genre_cols[:5]].head()

Filas originales:             114000
Tras limpieza básica:          113549
Canciones únicas (final):       89740


,track_id,track_name,popularity,genre_acoustic,genre_afrobeat,genre_alt-rock,genre_alternative,genre_ambient
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,73,1,0,0,0,0
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,55,1,0,0,0,0
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,57,1,0,0,0,0
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,71,1,0,0,0,0
4,5vjLSffimiIP26QG5WcN2K,Hold On,82,1,0,0,0,0


**Resultado:** 114.000 filas → **89.740 canciones únicas**, una fila por canción, con 114 columnas `genre_*` (multi-hot). El dataset queda en `data/processed/spotify_tracks_clean.csv` listo para el EDA y el modelamiento.